# 🔍 CholecSeg8k Label Analysis - Complete Class Mapping

**Purpose**: Systematically analyze CholecSeg8k watershed masks to identify:
1. All unique class labels (pixel values)
2. Which classes correspond to surgical instruments
3. Frequency distribution of each class
4. Sample visualizations for each class

**Dataset**: CholecSeg8k
- 8,080 annotated frames from 17 videos
- Watershed segmentation with multiple classes
- Need to identify: Blood (Class 24) + All Instrument Classes

In [1]:
# Cell 1: Imports
import numpy as np
import cv2
import matplotlib.pyplot as plt
from pathlib import Path
from collections import defaultdict, Counter
from tqdm import tqdm
import pandas as pd
import json

print("✅ Imports complete")

✅ Imports complete


In [2]:
# Cell 2: Setup Paths
CHOLECSEG_DIR = Path("cholecseg8k_raw")
OUTPUT_DIR = Path("label_analysis")
OUTPUT_DIR.mkdir(exist_ok=True)

print(f"Dataset directory: {CHOLECSEG_DIR}")
print(f"Output directory: {OUTPUT_DIR}")

# Find all available videos
videos = sorted([d for d in CHOLECSEG_DIR.iterdir() if d.is_dir()])
print(f"\nFound {len(videos)} videos:")
for vid in videos[:5]:
    print(f"  - {vid.name}")
if len(videos) > 5:
    print(f"  ... and {len(videos) - 5} more")

Dataset directory: cholecseg8k_raw
Output directory: label_analysis

Found 17 videos:
  - video01
  - video09
  - video12
  - video17
  - video18
  ... and 12 more


In [3]:
# Cell 3: Scan All Masks for Unique Labels
print("Scanning all masks for unique labels...\n")

all_labels = set()
label_counts = Counter()
video_label_map = defaultdict(set)

for video_dir in tqdm(videos, desc="Scanning videos"):
    mask_dir = video_dir / "watershed_mask"
    
    if not mask_dir.exists():
        continue
    
    mask_files = list(mask_dir.glob("*.png"))
    sample_size = min(50, len(mask_files))
    sampled_masks = np.random.choice(mask_files, sample_size, replace=False)
    
    for mask_path in sampled_masks:
        mask = cv2.imread(str(mask_path), cv2.IMREAD_GRAYSCALE)
        if mask is None:
            continue
        
        unique_labels = np.unique(mask)
        all_labels.update(unique_labels)
        
        for label in unique_labels:
            label_counts[label] += np.sum(mask == label)
            video_label_map[label].add(video_dir.name)

all_labels = sorted(all_labels)

print(f"\n✅ Scan complete!")
print(f"   Total unique labels found: {len(all_labels)}")
print(f"   Label range: {min(all_labels)} to {max(all_labels)}")

Scanning all masks for unique labels...



Scanning videos: 100%|████████████████████████████████████████████████████████████████████████████| 17/17 [00:00<00:00, 47220.64it/s]


✅ Scan complete!
   Total unique labels found: 0


ValueError: min() iterable argument is empty

In [4]:
# Cell 4: Create Frequency Table
label_data = []
for label in all_labels:
    label_data.append({
        'Label': label,
        'Total Pixels': label_counts[label],
        'Videos': len(video_label_map[label]),
        'Percentage': 100 * label_counts[label] / sum(label_counts.values())
    })

df = pd.DataFrame(label_data)
df = df.sort_values('Total Pixels', ascending=False)

print("Top 20 Most Frequent Labels:")
print(df.head(20).to_string(index=False))

df.to_csv(OUTPUT_DIR / "label_frequency_table.csv", index=False)
print(f"\n📊 Full table saved: {OUTPUT_DIR / 'label_frequency_table.csv'}")

KeyError: 'Total Pixels'

In [ ]:
# Cell 5: Known Class Mappings
KNOWN_CLASSES = {
    0: "Background",
    11: "Abdominal Wall",
    12: "Liver",
    13: "Gastrointestinal Tract",
    14: "Fat",
    15: "Grasping Retractor",
    21: "Grasper (Instrument)",
    22: "Bipolar (Instrument)",
    23: "Hook (Instrument)",
    24: "Blood",
    25: "Scissors (Instrument)",
    26: "Clipper (Instrument)",
    27: "Irrigator (Instrument)",
    28: "Specimen Bag",
    31: "Suction Instrument",
    32: "Hepatic Vein",
    33: "Liver Ligament",
}

INSTRUMENT_CLASSES = {
    21: "Grasper",
    22: "Bipolar",
    23: "Hook",
    25: "Scissors",
    26: "Clipper",
    27: "Irrigator",
    28: "Specimen Bag",
    31: "Suction"
}

print("Known Surgical Instruments:")
for label in sorted(INSTRUMENT_CLASSES.keys()):
    status = "✓ FOUND" if label in all_labels else "✗ NOT FOUND"
    print(f"  Class {label:2d}: {INSTRUMENT_CLASSES[label]:20s} - {status}")

print("\nBlood:")
status = "✓ FOUND" if 24 in all_labels else "✗ NOT FOUND"
print(f"  Class 24: Blood - {status}")

In [ ]:
# Cell 6: Visualize Sample Masks
def find_sample_with_class(video_dirs, target_class, max_search=100):
    """Find a sample frame that contains the target class"""
    for video_dir in video_dirs:
        mask_dir = video_dir / "watershed_mask"
        image_dir = video_dir / "endo"
        
        if not mask_dir.exists() or not image_dir.exists():
            continue
        
        mask_files = list(mask_dir.glob("*.png"))
        np.random.shuffle(mask_files)
        
        for mask_path in mask_files[:max_search]:
            mask = cv2.imread(str(mask_path), cv2.IMREAD_GRAYSCALE)
            if mask is None:
                continue
            
            if target_class in np.unique(mask):
                frame_name = mask_path.stem.replace("_watershed_mask", "")
                image_path = image_dir / f"{frame_name}.png"
                
                if image_path.exists():
                    image = cv2.imread(str(image_path))
                    image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
                    return image, mask, str(mask_path)
    
    return None, None, None

print("Searching for sample frames...")

In [ ]:
# Cell 7: Generate Visualizations for Each Instrument
for class_id, class_name in INSTRUMENT_CLASSES.items():
    if class_id not in all_labels:
        print(f"✗ {class_name} (Class {class_id}): Not found")
        continue
    
    image, mask, mask_path = find_sample_with_class(videos, class_id, max_search=200)
    
    if image is None:
        print(f"✗ {class_name} (Class {class_id}): No sample found")
        continue
    
    fig, axes = plt.subplots(1, 3, figsize=(15, 5))
    
    axes[0].imshow(image)
    axes[0].set_title('Original Frame')
    axes[0].axis('off')
    
    class_mask = (mask == class_id).astype(np.uint8) * 255
    axes[1].imshow(class_mask, cmap='hot')
    axes[1].set_title(f'{class_name} Mask (Class {class_id})')
    axes[1].axis('off')
    
    overlay = image.copy()
    overlay[mask == class_id] = overlay[mask == class_id] * 0.5 + np.array([255, 0, 0]) * 0.5
    axes[2].imshow(overlay.astype(np.uint8))
    axes[2].set_title('Overlay')
    axes[2].axis('off')
    
    plt.suptitle(f'{class_name} (Class {class_id})', fontsize=16, fontweight='bold')
    plt.tight_layout()
    
    output_path = OUTPUT_DIR / f"class_{class_id:02d}_{class_name.lower()}_sample.png"
    plt.savefig(output_path, dpi=150, bbox_inches='tight')
    plt.show()
    
    print(f"✓ {class_name} (Class {class_id}): Saved")

In [ ]:
# Cell 8: Save Class Mapping
INSTRUMENT_MAPPING = {
    class_id: class_name 
    for class_id, class_name in INSTRUMENT_CLASSES.items() 
    if class_id in all_labels
}

if 24 in all_labels:
    INSTRUMENT_MAPPING[24] = "Blood"

print("Final Class Mapping:")
for class_id, class_name in sorted(INSTRUMENT_MAPPING.items()):
    print(f"  {class_id}: {class_name}")

# Save as JSON
mapping_path = OUTPUT_DIR / "class_mapping.json"
with open(mapping_path, 'w') as f:
    json.dump(INSTRUMENT_MAPPING, f, indent=2)

print(f"\n💾 Saved: {mapping_path}")

# Save as Python dict
dict_path = OUTPUT_DIR / "class_mapping.py"
with open(dict_path, 'w') as f:
    f.write("# CholecSeg8k Class Mapping\n\n")
    f.write("CLASS_MAPPING = {\n")
    for class_id, class_name in sorted(INSTRUMENT_MAPPING.items()):
        f.write(f"    {class_id}: '{class_name}',\n")
    f.write("}\n")

print(f"💾 Saved: {dict_path}")

In [ ]:
# Cell 9: Summary Report
print("\n" + "="*80)
print("SUMMARY REPORT")
print("="*80)

print(f"\nTotal unique labels: {len(all_labels)}")
print(f"Instruments found: {len([c for c in INSTRUMENT_CLASSES.keys() if c in all_labels])}/{len(INSTRUMENT_CLASSES)}")
print(f"Blood (Class 24): {'✓ Found' if 24 in all_labels else '✗ Not found'}")

print("\nGenerated Files:")
print(f"  1. label_frequency_table.csv")
print(f"  2. class_mapping.json")
print(f"  3. class_mapping.py")
print(f"  4. Sample visualizations: {len(list(OUTPUT_DIR.glob('class_*.png')))} images")

print("\nNext Steps:")
print("  1. Review sample visualizations")
print("  2. Proceed to 03_instrument_segmentation.ipynb")
print("  3. Use class_mapping.json for training")